# RoBERTa NLI Optimised (Google Colab)

**Notebook sections:** **§1 Training** → **§2 Evaluation (dev)** → **§3 Test inference (demo)**.

Improvements over the baseline:
1. **Data path**: Google Drive (e.g. upload `train.csv` and `dev.csv`; adjust `_data_dir` in the data cell)
2. **MNLI checkpoint** + dev-based checkpoint selection (`eval_macro_f1`)
3. **Test inference**: `roberta_predictions.csv` when `test.csv` is available (§3; see prerequisites there)


## Dependencies

```bash
pip install transformers datasets evaluate accelerate scikit-learn
# Upgrade if needed: pip install --upgrade transformers
```

Run the **install** cell below first. On **Google Colab**, use **GPU** runtime if possible. Mount Google Drive in the next cell if your data and saved model live under Drive.


In [9]:
# Install dependencies (run this cell first)
%pip -q install transformers datasets evaluate accelerate scikit-learn
%pip install --upgrade transformers


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Training

Load `train.csv` / `dev.csv`, clean text, build Datasets, tokenize, load `prajjwal1/roberta-base-mnli`, create `Trainer`, then **`trainer.train()`** and **`trainer.save_model`**.

**Outputs:** checkpoints under `OUTPUT_DIR` (default `/content/roberta_nli_output` on Colab); you can also copy/save under `_data_dir` (e.g. Drive) for the demo section.


In [4]:
import pandas as pd
import numpy as np
import os
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# Colab: data path (adjust if using Drive)
_data_dir = "/content/drive/MyDrive/nlu_cwk_data"
TRAIN_PATH = f"{_data_dir}/train.csv"
DEV_PATH = f"{_data_dir}/dev.csv"

train_df = pd.read_csv(TRAIN_PATH)
dev_df = pd.read_csv(DEV_PATH)
print(f"Data path: {_data_dir}, train: {len(train_df)}, dev: {len(dev_df)}")


Data path: /content/drive/MyDrive/nlu_cwk_data, train: 24432, dev: 6736


In [7]:
import re
import unicodedata
import html

def clean_for_roberta(text):
    """Text cleaning for RoBERTa: preserve semantics, reduce noise."""
    if pd.isna(text):
        return ""
    text = str(text)

    # 1. HTML entity decoding (&amp;, &quot;, etc.)
    text = html.unescape(text)

    # 2. Unicode normalisation (NFKC; e.g. ligatures)
    text = unicodedata.normalize("NFKC", text)

    # 3. Normalise quotes and dashes (fewer subword splits)
    text = text.replace(""", '"').replace(""", '"')
    text = text.replace("'", "'").replace("'", "'")
    text = text.replace("–", "-").replace("—", "-")

    # 4. Strip control characters (\x00-\x1f)
    text = "".join(ch for ch in text if ch not in "\x00\x01\x02\x03\x04\x05\x06\x07\x08\x0b\x0c\x0e\x0f\x10\x11\x12\x13\x14\x15\x16\x17\x18\x19\x1a\x1b\x1c\x1d\x1e\x1f")
    text = text.replace("\r\n", " ").replace("\r", " ").replace("\n", " ")
    text = text.replace("\t", " ")

    # 5. Normalise whitespace
    text = " ".join(text.split())

    return text.strip()

PREMISE_COL = "premise"
HYPOTHESIS_COL = "hypothesis"
LABEL_COL = "label"

train_df[PREMISE_COL] = train_df[PREMISE_COL].apply(clean_for_roberta)
train_df[HYPOTHESIS_COL] = train_df[HYPOTHESIS_COL].apply(clean_for_roberta)
dev_df[PREMISE_COL] = dev_df[PREMISE_COL].apply(clean_for_roberta)
dev_df[HYPOTHESIS_COL] = dev_df[HYPOTHESIS_COL].apply(clean_for_roberta)


In [10]:
# Build Hugging Face Datasets
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[[PREMISE_COL, HYPOTHESIS_COL, LABEL_COL]].rename(columns={LABEL_COL: "label"})
)
dev_ds = Dataset.from_pandas(
    dev_df[[PREMISE_COL, HYPOTHESIS_COL, LABEL_COL]].rename(columns={LABEL_COL: "label"})
)

print(train_ds)
print(dev_ds)


Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 24432
})
Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 6736
})


In [11]:
# Load tokenizer
from transformers import AutoTokenizer
MODEL_NAME = "prajjwal1/roberta-base-mnli"
TOKENIZER_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True   # replace classifier head; load encoder weights
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: prajjwal1/roberta-base-mnli
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH:	ckpt wei

In [12]:
# tokenization
MAX_LENGTH = 256

def tokenize_function(examples):
    return tokenizer(
        examples[PREMISE_COL],
        examples[HYPOTHESIS_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_ds.map(tokenize_function, batched=True)
dev_tokenized = dev_ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

In [ ]:
# Load model (MNLI-pretrained; num_labels=2 binary head)
from transformers import AutoModelForSequenceClassification

num_labels = len(sorted(train_df[LABEL_COL].unique()))  # 2

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,  # binary 0/1 task; replaces 3-way head
    ignore_mismatched_sizes=True   # allow head size mismatch; load encoder
)


You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: prajjwal1/roberta-base-mnli
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH:	ckpt wei

In [ ]:
# Evaluation metrics
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }


In [ ]:
# Training arguments (early stopping optional)
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

OUTPUT_DIR = "/content/roberta_nli_output"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,  # upper bound; early stopping can stop sooner
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    fp16=True
)


In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=dev_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.275832,0.260964,0.898753,0.898476
2,0.207135,0.379396,0.905285,0.905029
3,0.130925,0.479996,0.903800,0.903599
4,0.069771,0.556115,0.907660,0.907507
5,0.046816,0.601484,0.907067,0.906899


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=7635, training_loss=0.1549806936472635, metrics={'train_runtime': 845.5025, 'train_samples_per_second': 144.482, 'train_steps_per_second': 9.03, 'total_flos': 4877552744263680.0, 'train_loss': 0.1549806936472635, 'epoch': 5.0})

## 2. Evaluation (development set)

Runs **`trainer.predict(dev_tokenized)`** on the course `dev.csv` (metrics printed below). This is **separate** from training and from §3 test inference.


In [ ]:

predictions = trainer.predict(dev_tokenized)
print("Eval results:", predictions.metrics)

Eval results: {'test_loss': 0.5560081005096436, 'test_accuracy': 0.9079572446555819, 'test_macro_f1': 0.907804696920127, 'test_runtime': 8.0286, 'test_samples_per_second': 839.002, 'test_steps_per_second': 26.281}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

trainer.save_model(OUTPUT_DIR)
MODEL_PATH = OUTPUT_DIR
tokenizer_test = AutoTokenizer.from_pretrained("roberta-base")
model_test = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 3. Test inference (demo)

Loads a **saved** fine-tuned model from disk (same layout as after `trainer.save_model` or a `checkpoint-*` folder) — **no in-memory `trainer` required**.

### Before running this cell (kernel restart or demo-only)

Run **all** of the following **in order** so that names exist in the kernel:

1. **Dependencies:** install cell + (optional) Drive mount.
2. **§1 — up to and including** the cell that defines **`clean_for_roberta`**, **`PREMISE_COL` / `HYPOTHESIS_COL`**, and **`MAX_LENGTH`** (i.e. run the **data**, **cleaning**, **datasets**, **tokenizer**, and **tokenization** cells — through the cell that builds `train_tokenized` / `dev_tokenized` and sets `MAX_LENGTH`).
3. You do **not** need to re-run **`trainer.train()`** if you already have weights on disk; but you **must** have run the cleaning/tokenizer cells **or** redefine the same constants manually.

If you skip step 2, you may see `NameError: PREMISE_COL` (or `clean_for_roberta`, `MAX_LENGTH`).

**Test file:** `test.csv` is resolved via `_data_dir/test.csv`, or `test_data/NLI/test.csv`, or `../test_data/NLI/test.csv`.

**Model path:** tries `OUTPUT_DIR`, `/content/roberta_nli_output`, `optimisation/roberta_nli_output`, then `trainer_state.json` best checkpoint or a `checkpoint-*` folder.


In [13]:
# === Test inference (demo): load saved weights from disk; no `trainer` in memory required ===
import glob
import json

OUTPUT_DIR_DEFAULT = "/content/drive/MyDrive/nlu_cwk_data/roberta_nli_output"


def _has_model_files(p):
    return os.path.isdir(p) and os.path.isfile(os.path.join(p, "config.json"))


def _resolve_saved_model_dir():
    """Directory with config.json: output root, best checkpoint, or first checkpoint-*."""
    roots = []
    if "OUTPUT_DIR" in globals() and OUTPUT_DIR:
        roots.append(OUTPUT_DIR)
    roots.extend(
        [
            OUTPUT_DIR_DEFAULT,
            os.path.join("optimisation", "roberta_nli_output"),
            os.path.join("..", "optimisation", "roberta_nli_output"),
        ]
    )
    seen = set()
    for root in roots:
        if not root or root in seen:
            continue
        seen.add(root)
        if _has_model_files(root):
            return root
        state_path = os.path.join(root, "trainer_state.json")
        if os.path.isfile(state_path):
            try:
                with open(state_path, encoding="utf-8") as f:
                    st = json.load(f)
                best = st.get("best_model_checkpoint")
                if best and _has_model_files(best):
                    return best
            except (OSError, json.JSONDecodeError):
                pass
        for ckpt in sorted(glob.glob(os.path.join(root, "checkpoint-*"))):
            if _has_model_files(ckpt):
                return ckpt
    return None


MODEL_PATH = _resolve_saved_model_dir()
if MODEL_PATH is None:
    print(
        "No saved model found. Train and run save_model, or place weights under "
        "optimisation/roberta_nli_output/ (e.g. unzip OneDrive) or Colab /content/roberta_nli_output."
    )

_test_candidates = []
if "_data_dir" in globals() and _data_dir:
    _test_candidates.append(os.path.join(_data_dir, "test.csv"))
_test_candidates.extend(
    [
        "test_data/NLI/test.csv",
        "../test_data/NLI/test.csv",
    ]
)
test_path = next((p for p in _test_candidates if os.path.exists(p)), None)

if MODEL_PATH is None:
    pass
elif not test_path:
    print("test.csv not found; tried:", _test_candidates)
else:
    try:
        tok_infer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    except Exception:
        tok_infer = AutoTokenizer.from_pretrained("roberta-base")
    model_infer = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, local_files_only=True
    )
    infer_args = TrainingArguments(
        output_dir="./.tmp_roberta_infer",
        per_device_eval_batch_size=32,
        report_to="none",
    )
    trainer_infer = Trainer(
        model=model_infer,
        args=infer_args,
        processing_class=tok_infer,
    )
    print(f"Loading model from: {MODEL_PATH}")
    print(f"Loading test set: {test_path}")
    test_df = pd.read_csv(test_path)
    test_df[PREMISE_COL] = test_df[PREMISE_COL].apply(clean_for_roberta)
    test_df[HYPOTHESIS_COL] = test_df[HYPOTHESIS_COL].apply(clean_for_roberta)
    test_ds = Dataset.from_pandas(test_df[[PREMISE_COL, HYPOTHESIS_COL]])
    test_tokenized = test_ds.map(
        lambda ex: tok_infer(
            ex[PREMISE_COL],
            ex[HYPOTHESIS_COL],
            truncation=True,
            max_length=MAX_LENGTH,
        ),
        batched=True,
    )
    test_tokenized = test_tokenized.remove_columns([PREMISE_COL, HYPOTHESIS_COL])
    preds = trainer_infer.predict(test_tokenized)
    labels = np.argmax(preds.predictions, axis=-1)
    pd.DataFrame({"label": labels}).to_csv("roberta_predictions.csv", index=False)
    print("Saved roberta_predictions.csv")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading model from: /content/drive/MyDrive/nlu_cwk_data/roberta_nli_output
Loading test set: /content/drive/MyDrive/nlu_cwk_data/test.csv


Map:   0%|          | 0/3302 [00:00<?, ? examples/s]

Saved roberta_predictions.csv
